# 🚀 Vecna / AIC51: Trích xuất Qwen-VL, ASR, OCR từ `keyframes.rar` trên Drive

Notebook này được tối ưu khi bạn **đã có sẵn file `keyframes.rar` trên Google Drive**:
1. **Giải nén `keyframes.rar` từ Drive** thẳng vào ổ SSD local của Colab (`/content/workspace/data/keyframes/`).
2. **Tách Audio (nếu cần ASR)**: Tải nhanh video hoặc dùng `audio.rar` từ Drive để phục vụ WhisperX.
3. **Trích xuất đa phương thái** bằng `aic51-cli analyse` (Có log thời gian thực realtime):
   - 🧠 **Qwen-VL Embedding** (`qwen_vl.npy` từ keyframes)
   - 📝 **OCR Text Recognition** (`ocr.npy` từ keyframes qua PaddleVietOCR)
   - 🎙️ **ASR Speech-to-Text** (`asr.npy` qua WhisperX nếu có audio)
4. **Lưu trữ Features vĩnh viễn** sang Google Drive và kiểm tra tính toàn vẹn.

### 1. Mount Google Drive & Kiểm tra GPU

In [ ]:
# 1. Mount Drive (Nơi chứa keyframes.rar và lưu trữ kết quả)
from google.colab import drive
import os

drive.mount('/content/drive')

# Kiểm tra thông số GPU (Khuyến nghị chọn A100 GPU trên Colab Pro+)
!nvidia-smi

### 2. Cài đặt Dependencies Hệ thống & Clone Vecna Repo

In [ ]:
%%bash
# Cài đặt unrar (để giải nén keyframes.rar), aria2, ffmpeg và Tesseract OCR
apt-get update -qq
apt-get install -y -qq unrar aria2 ffmpeg tesseract-ocr tesseract-ocr-vie

# Clone source code dự án Vecna
cd /content
if [ ! -d "/content/Vecna" ]; then
    git clone https://github.com/nlmhoagn/Vecna.git /content/Vecna
    # Hoặc: git clone https://github.com/JimmyK300/Vecna.git /content/Vecna
fi

### 3. Cài đặt Python Dependencies (PaddleOCR, VietOCR, WhisperX, Qwen-VL)

In [ ]:
%%bash
# 1. Cài đặt aic51 CLI từ source
cd /content/Vecna/aic51-src
pip install -q -e .

# 2. Cài đặt PaddleOCR & VietOCR cho tác vụ OCR tiếng Việt
pip install -q paddlepaddle paddleocr vietocr

# 3. Cài đặt WhisperX và CTranslate2 cho tác vụ ASR
pip install -q whisperx --no-deps
pip install -q faster-whisper "ctranslate2>=4.5.0" pyannote.audio nltk pandas

# 4. Cài đặt Sentence-Transformers & phụ trợ
pip install -q sentence-transformers open_clip_torch deep-translator pymilvus

### 4. (Tuỳ chọn) Tải Audio cho ASR WhisperX

> **Lưu ý:**  
> - **Qwen-VL và OCR** chỉ cần ảnh trong `keyframes.rar` là chạy được ngay.  
> - **ASR (nhận diện giọng nói)** bắt buộc cần file âm thanh `.wav`.  
> - Nếu bạn chỉ cần Qwen và OCR, bạn có thể **bỏ qua Cell 4 và Cell 6**.  
> - Nếu cần cả ASR, hãy chạy Cell này để tải nhanh 2 video batch M09/M10 tách lấy audio.

In [ ]:
import os
import subprocess

# Chỉ chạy nếu bạn muốn bóc audio cho ASR WhisperX từ video gốc
os.makedirs("/content/zips", exist_ok=True)
os.makedirs("/content/raw_videos", exist_ok=True)

batches = [
    ("Videos_M09.zip", "https://aic-data.ledo.io.vn/Videos_M09.zip"),
    ("Videos_M10.zip", "https://aic-data.ledo.io.vn/Videos_M10.zip")
]

for filename, url in batches:
    zip_path = f"/content/zips/{filename}"
    print(f"[*] Đang tải {filename} để lấy audio...")
    subprocess.run(["aria2c", "-x", "16", "-s", "16", "-d", "/content/zips", "-o", filename, url], check=True)
    subprocess.run(["unzip", "-q", "-o", zip_path, "-d", "/content/raw_videos/"], check=True)
    os.remove(zip_path)

print("Hoàn tất chuẩn bị video cho audio!")

### 5. Khởi tạo Workspace Vecna & Đồng bộ Config

In [ ]:
%%bash
mkdir -p /content/workspace
cd /content/workspace

# Khởi tạo cấu trúc layout workspace
aic51-cli init

# Đồng bộ file config.yaml từ repo Vecna
if [ -f "/content/Vecna/config.yaml" ]; then
    cp /content/Vecna/config.yaml /content/workspace/config.yaml
    echo "[✓] Đã đồng bộ config.yaml vào workspace"
fi

### 6. (Tuỳ chọn cho ASR) Tách Audio từ Video vào `data/audio/`

In [ ]:
import subprocess
from pathlib import Path

workspace_dir = "/content/workspace"
raw_videos_dir = Path("/content/raw_videos")

# Nếu có video tải về ở Cell 4, chạy lệnh tách audio (cờ -a)
if raw_videos_dir.exists() and any(raw_videos_dir.rglob("*.mp4")):
    video_folders = set(f.parent for f in raw_videos_dir.rglob("*.mp4"))
    for folder in sorted(video_folders):
        print(f"[*] Đang tách audio từ: {folder.name}")
        subprocess.run(
            ["aic51-cli", "add", str(folder), "-d", "-a"],
            cwd=workspace_dir,
            check=True
        )
    # Xoá video thô sau khi đã lấy được audio để tiết kiệm ổ đĩa
    !rm -rf /content/raw_videos/*
    print("[✓] Đã tách xong audio vào data/audio/!")
else:
    print("Bỏ qua tách audio từ video (không có video thô hoặc bạn đã có audio riêng).")

### 7. Giải nén `keyframes.rar` từ Google Drive vào Workspace Local

- Cell này sẽ tự động tìm kiếm file `keyframes.rar` (hoặc `keyframes.zip`) trên Google Drive của bạn.
- Sau đó giải nén thẳng vào `/content/workspace/data/keyframes/` và tự động chuẩn hoá cấu trúc thư mục (dạng `data/keyframes/{video_id}/{frame_id}.jpg`).

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

target_kf_dir = Path("/content/workspace/data/keyframes")
target_kf_dir.mkdir(parents=True, exist_ok=True)

# 1. Tìm vị trí file keyframes.rar trên Drive
rar_candidates = list(Path("/content/drive/MyDrive").rglob("keyframes.rar"))
if not rar_candidates:
    rar_candidates = list(Path("/content/drive/MyDrive").rglob("*keyframe*.rar")) + list(Path("/content/drive/MyDrive").rglob("*keyframe*.zip"))

if not rar_candidates:
    raise FileNotFoundError("Chưa tìm thấy file keyframes.rar trên Google Drive (/content/drive/MyDrive/). Hãy đảm bảo bạn đã tải file lên Drive!")

rar_path = rar_candidates[0]
print(f"[✓] Đã tìm thấy file keyframes trên Drive: {rar_path}")

# 2. Giải nén vào thư mục tạm
temp_extract = Path("/content/temp_keyframes")
temp_extract.mkdir(exist_ok=True)

print("[*] Đang giải nén keyframes từ Google Drive sang SSD Colab...")
if rar_path.suffix.lower() == ".rar":
    subprocess.run(["unrar", "x", "-o+", str(rar_path), str(temp_extract)], check=True)
else:
    subprocess.run(["unzip", "-q", "-o", str(rar_path), "-d", str(temp_extract)], check=True)

# 3. Chuẩn hoá cấu trúc thư mục vào /content/workspace/data/keyframes/{video_id}/
subdirs = [d for d in temp_extract.iterdir() if d.is_dir()]
if len(subdirs) == 1 and subdirs[0].name.lower() in ["keyframes", "data"]:
    source_dir = subdirs[0]
else:
    source_dir = temp_extract

for item in source_dir.iterdir():
    if item.is_dir():
        dest = target_kf_dir / item.name
        if dest.exists():
            shutil.rmtree(dest)
        shutil.move(str(item), str(dest))

shutil.rmtree(temp_extract, ignore_errors=True)

# Thống kê kết quả giải nén
videos = list(target_kf_dir.glob("*"))
total_images = sum(1 for _ in target_kf_dir.rglob("*.jpg"))
print(f"\n=======================================================")
print(f"[✓] Giải nén thành công {len(videos)} video ({total_images} ảnh keyframe) vào /content/workspace/data/keyframes/")
print(f"Danh sách một số video: {[v.name for v in videos[:10]]}")

### 8. Chạy `aic51-cli analyse` xuất Qwen & OCR (Kèm Log Thời Gian Thực Real-time)

- Hiển thị mốc thời gian `[HH:MM:SS]` và tổng thời gian đã trôi qua `(+MMm SSs)` cho từng dòng log.
- Tự động theo dõi tiến độ từng video, tốc độ (docs/s), và thời gian dự kiến còn lại (ETA).

In [ ]:
import os
import sys
import time
import subprocess
from pathlib import Path
from datetime import datetime

workspace_dir = "/content/workspace"
audio_dir = Path("/content/workspace/data/audio")
start_time = time.time()

def get_time_str():
    return datetime.now().strftime("%H:%M:%S")

def format_elapsed(seconds):
    mins, secs = divmod(int(seconds), 60)
    hours, mins = divmod(mins, 60)
    if hours > 0:
        return f"{hours:02d}h {mins:02d}m {secs:02d}s"
    return f"{mins:02d}m {secs:02d}s"

cmd = ["aic51-cli", "analyse", "--use-qwen-vl", "--use-ocr", "--keep-going"]

if audio_dir.exists() and any(audio_dir.glob("*.wav")):
    print(f"[{get_time_str()}] [✓] Tìm thấy audio -> Kích hoạt thêm ASR (WhisperX)!")
    cmd.insert(3, "--use-asr")
else:
    print(f"[{get_time_str()}] [*] Chạy Qwen-VL & OCR từ keyframes (không có audio)!")

print(f"[{get_time_str()}] 🚀 BẮT ĐẦU PIPELINE: {' '.join(cmd)}")
print("-" * 70)

# Thiết lập môi trường không buffer để hiển thị log tức thì từng giây
env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"

process = subprocess.Popen(
    cmd,
    cwd=workspace_dir,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=env
)

# Đọc stream log real-time và in kèm timestamp
for line in iter(process.stdout.readline, ""):
    line_clean = line.strip()
    if line_clean:
        elapsed = time.time() - start_time
        print(f"[{get_time_str()} | +{format_elapsed(elapsed)}] {line_clean}")
        sys.stdout.flush()

process.stdout.close()
return_code = process.wait()

total_time = time.time() - start_time
print("-" * 70)
if return_code == 0:
    print(f"[{get_time_str()}] [✓] HOÀN TẤT TRÍCH XUẤT! Tổng thời gian chạy: {format_elapsed(total_time)}")
else:
    print(f"[{get_time_str()}] [!] Quá trình dừng với mã {return_code}. Tổng thời gian: {format_elapsed(total_time)}")

### 9. Sao lưu Features lên Drive & Kiểm tra Toàn diện (Kèm Thống Kê & Thời Gian)

- Hiển thị thời gian nén và kích thước file lưu trữ trên Google Drive.
- Kiểm tra mẫu dữ liệu Qwen và OCR.

In [ ]:
import os
import sys
import time
import subprocess
import numpy as np
from pathlib import Path
from datetime import datetime

def get_time_str():
    return datetime.now().strftime("%H:%M:%S")

def format_elapsed(seconds):
    mins, secs = divmod(int(seconds), 60)
    hours, mins = divmod(mins, 60)
    if hours > 0:
        return f"{hours:02d}h {mins:02d}m {secs:02d}s"
    return f"{mins:02d}m {secs:02d}s"

workspace_dir = "/content/workspace"
drive_backup = "/content/drive/MyDrive/Vecna_M09_M10_Output"
os.makedirs(drive_backup, exist_ok=True)
features_dir = Path("/content/workspace/features")
keyframes_dir = Path("/content/workspace/data/keyframes")

step_start = time.time()
print(f"[{get_time_str()}] 📦 BẮT ĐẦU ĐÓNG GÓI & SAO LƯU SANG GOOGLE DRIVE...")

# 1. Thống kê số lượng file
qwen_files = list(features_dir.rglob("qwen_vl.npy"))
ocr_files = list(features_dir.rglob("ocr.npy"))
total_videos = len(list(features_dir.glob("M09*")) + list(features_dir.glob("M10*")))

print(f"[{get_time_str()}] Thống kê dữ liệu đã trích xuất:")
print(f"   - Tổng số video: {total_videos}")
print(f"   - Tổng số vector Qwen: {len(qwen_files):,} file")
print(f"   - Tổng số text OCR: {len(ocr_files):,} file")

# 2. Đóng gói sang Google Drive với đo đạc thời gian nén
tar_start = time.time()
features_tar = os.path.join(drive_backup, "features_M09_M10.tar.gz")
print(f"[{get_time_str()}] Đang nén thư mục features sang: {features_tar} ...")

subprocess.run(["tar", "-czf", features_tar, "-C", workspace_dir, "features"], check=True)

tar_elapsed = time.time() - tar_start
tar_size_mb = os.path.getsize(features_tar) / (1024 * 1024)
print(f"[{get_time_str()}] [✓] Nén & lưu xong sau {format_elapsed(tar_elapsed)}! Kích thước file: {tar_size_mb:.2f} MB")

# 3. Đọc thử mẫu dữ liệu để xác nhận
print(f"\n[{get_time_str()}] 🔍 ĐANG KIỂM TRA MẪU DỮ LIỆU ĐÃ XUẤT:")
sample_videos = sorted(list(features_dir.glob("M09*")) + list(features_dir.glob("M10*")))
if sample_videos:
    sample_vid = sample_videos[0]
    sample_frames = sorted(list(sample_vid.glob("*")))
    if sample_frames:
        frame_dir = sample_frames[0]
        print(f" - Video mẫu: {sample_vid.name}, Frame mẫu: {frame_dir.name}")
        
        ocr_file = frame_dir / "ocr.npy"
        if ocr_file.exists():
            ocr_text = np.load(ocr_file, allow_pickle=True)
            print(f"   [OCR Text]: \"{str(ocr_text).strip()}\"")
            
        qwen_file = frame_dir / "qwen_vl.npy"
        if qwen_file.exists():
            vec = np.load(qwen_file)
            print(f"   [Qwen Vector]: Shape={vec.shape}, Dtype={vec.dtype}")

total_step_time = time.time() - step_start
print(f"\n=======================================================")
print(f"[{get_time_str()}] 🎉 TOÀN BỘ TIẾN TRÌNH HOÀN TẤT THÀNH CÔNG!")
print(f"[{get_time_str()}] Thời gian đóng gói & kiểm tra: {format_elapsed(total_step_time)}")
print(f"[{get_time_str()}] File lưu vĩnh viễn trên Drive: {features_tar}")
print(f"=======================================================")